# Practical 5 - Understanding Two Tailed Test

First two questions use the teachers' rating dataset again to work out some probabilities around the evaluation scores. The third question moves away from that dataset and works through a two-tailed hypothesis test using a basketball scoring example.

Loading the libraries and the dataset first.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("TeachingRatings.csv")
df.head()

,rownames,minority,age,gender,credits,beauty,eval,division,native,tenure,students,allstudents,prof
0,1,yes,36,female,more,0.289916,4.3,upper,yes,yes,24,43,1
1,2,no,59,male,more,-0.737732,4.5,upper,yes,yes,17,20,2
2,3,no,51,male,more,-0.571984,3.7,upper,yes,yes,55,55,3
3,4,no,40,female,more,-0.677963,4.3,upper,yes,yes,40,46,4
4,5,no,31,female,more,1.509794,4.4,upper,yes,yes,42,48,5


### Q1. Probability of receiving an evaluation score greater than 4.5

To work this out properly, I'm treating the eval scores as roughly normally distributed, using the mean and standard deviation from the data, then finding the probability of scoring above 4.5 using the normal distribution. I'm also checking the actual proportion of professors who scored above 4.5 in the real data, just to compare.

In [2]:
mean_eval = df["eval"].mean()
std_eval = df["eval"].std()
print("Mean evaluation score:", round(mean_eval, 3))
print("Std dev:", round(std_eval, 3))

# Using the normal distribution
prob_above_45 = 1 - stats.norm.cdf(4.5, loc=mean_eval, scale=std_eval)
print("\nP(eval > 4.5) using normal distribution:", round(prob_above_45, 4))

# Comparing to the actual data
empirical_prob = (df["eval"] > 4.5).mean()
print("Actual proportion in the data scoring above 4.5:", round(empirical_prob, 4))

Mean evaluation score: 3.998
Std dev: 0.555

P(eval > 4.5) using normal distribution: 0.1829
Actual proportion in the data scoring above 4.5: 0.1706


**Answer:** Assuming the evaluation scores follow a normal distribution, the probability of getting a score above 4.5 is about **18.3%**. The actual proportion of professors in the dataset who scored above 4.5 is close to that, around 17.1%, so the normal approximation fits the real data reasonably well here.

### Q2. Probability of receiving a score greater than 3.5 and less than 4.2

Same idea, just for a range this time instead of a single cutoff.

In [3]:
prob_between = stats.norm.cdf(4.2, loc=mean_eval, scale=std_eval) - stats.norm.cdf(3.5, loc=mean_eval, scale=std_eval)
print("P(3.5 < eval < 4.2) using normal distribution:", round(prob_between, 4))

empirical_between = ((df["eval"] > 3.5) & (df["eval"] < 4.2)).mean()
print("Actual proportion in the data between 3.5 and 4.2:", round(empirical_between, 4))

P(3.5 < eval < 4.2) using normal distribution: 0.4573
Actual proportion in the data between 3.5 and 4.2: 0.3542


**Answer:** Based on the normal distribution, the probability of a score falling between 3.5 and 4.2 is about **45.7%**. The actual data shows a somewhat lower proportion, around 35.4%, in that exact range. The gap here is a bit bigger than in Q1 — probably because the real eval scores aren't perfectly normal (they're a bit skewed, as we saw back in Practical 2), so the normal distribution isn't as precise a fit for this particular range.

### Q3. Two-tailed test comparing the pro team to regional players

This one's a hypothesis test, not about the ratings dataset. Here's the situation: pro players are known to average 12 points per game with a standard deviation of 5.5. A group of 36 regional players averaged 10.7 points per game, and the coach wants to know if regional players score differently (not just lower, but differently in either direction) from the historic pro average.

**Null hypothesis (H0):** The mean points of the regional players is not different from the historic mean (12).

**Alternative hypothesis (H1):** The mean points of the regional players is different from the historic mean.

Since we know the population standard deviation, this is a two-tailed z-test.

In [4]:
pop_mean = 12
pop_std = 5.5
n = 36
sample_mean = 10.7

# Standard error
se = pop_std / np.sqrt(n)

# z statistic
z_stat = (sample_mean - pop_mean) / se

# two-tailed p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("Standard error:", round(se, 4))
print("Z statistic:", round(z_stat, 4))
print("Two-tailed p-value:", round(p_value, 4))

Standard error: 0.9167
Z statistic: -1.4182
Two-tailed p-value: 0.1561


**Answer:** The z statistic comes out to about **-1.42**, with a two-tailed p-value of about **0.156**. Since 0.156 is greater than the usual significance level of 0.05, we **fail to reject the null hypothesis**. In plain terms: even though the regional players averaged a bit lower (10.7 vs 12), the difference isn't big enough to say with confidence that regional players truly score differently from the pro historic average — it's plausible this gap is just due to normal sample variation. The chart above shows our z-score sits inside the "not significant" middle zone, well short of either red rejection region.

## Conclusion

For the ratings dataset, treating evaluation scores as roughly normal gave probability estimates that were in the right ballpark compared to the actual data, though not a perfect match — closer for the single cutoff in Q1, a bit further off for the range in Q2, likely because the real distribution has a slight skew. For the basketball question, the two-tailed test showed that the regional players' lower average (10.7 vs a historic pro mean of 12) wasn't statistically significant at the 5% level, so there isn't strong enough evidence to say their true performance differs from the pro historic average.